# Test: gf_lm()

Tests the extended gf_lm() that adds categorical x support and shuffle-safe
continuous x. Covers: categorical, continuous, shuffle, gf_coef chaining,
gf_shuffle_grid, standalone passthrough, and edge cases.

In [ ]:
library(coursekata)
source("../gf_lm_cat.R")   # needed by gf_shuffle_grid
source("../gf_coef.R")
source("../gf_shuffle_grid.R")
source("../gf_lm.R")        # shadows ggformula::gf_lm

options(repr.plot.width = 6, repr.plot.height = 4)

## Categorical x

In [ ]:
# Test 1: basic categorical x — should draw group-mean segments
gf_jitter(later_anxiety ~ condition, data = er, width = 0.1, alpha = 0.4) %>%
  gf_lm()

In [ ]:
# Test 2: three-level categorical x
Fingers3 <- droplevels(subset(Fingers, Year %in% c("1", "2", "3")))
gf_jitter(Thumb ~ Year, data = Fingers3, width = 0.1, alpha = 0.4) %>%
  gf_lm()
gf_jitter(Thumb ~ RaceEthnic, data = Fingers3, width = 0.1, alpha = 0.4) %>%
  gf_lm()

In [ ]:
# Test 3: shuffle + categorical x
# Segments must match the jittered points — NOT a fresh shuffle
#set.seed(7)
gf_jitter(shuffle(later_anxiety) ~ condition, data = er, width = 0.1, alpha = 0.4) %>%
  gf_lm()

In [ ]:
# Test 4: axis labels should stay as entered in the formula
# y-axis: "shuffle(later_anxiety)", x-axis: "condition" — NOT .gf_y / .gf_x
#set.seed(7)
gf_jitter(shuffle(later_anxiety) ~ condition, data = er, width = 0.1, alpha = 0.4) %>%
  gf_lm()

## Continuous x

In [ ]:
# Test 5: basic continuous x — should draw a regression line (same as before)
gf_point(Thumb ~ Height, data = Fingers, alpha = 0.4) %>%
  gf_lm()

In [ ]:
# Test 6: shuffle + continuous x
# Regression line must fit the shuffled points — NOT a fresh shuffle
#set.seed(7)
gf_jitter(shuffle(Thumb) ~ Height, data = Fingers, width = 0.5, alpha = 0.4) %>%
  gf_lm()


In [ ]:
# Test 7: axis labels should stay as entered
# y-axis: "shuffle(Thumb)", x-axis: "Height"
set.seed(7)
gf_jitter(shuffle(Thumb) ~ Height, data = Fingers, width = 0.5, alpha = 0.4) %>%
  gf_lm()

## Chained with gf_coef()

In [ ]:
# Test 8: categorical x + gf_coef
# Segments from gf_lm(), b0 line and b1 arrow from gf_coef()
gf_jitter(later_anxiety ~ condition, data = er, width = 0.1, alpha = 0.4) %>%
  gf_lm() %>%
  gf_coef()

In [ ]:
# Test 9: shuffle + categorical x + gf_coef
# Points, segments, and arrows must all reflect the same shuffle
#set.seed(7)
gf_jitter(shuffle(later_anxiety) ~ condition, data = er, width = 0.1, alpha = 0.4) %>%
  gf_lm() %>%
  gf_coef()

In [ ]:
# Test 10: continuous x + gf_coef
# Rise-over-run annotation from gf_coef(), regression line from gf_lm()
gf_point(Thumb ~ Height, data = Fingers, alpha = 0.4) %>%
  gf_lm() %>%
  gf_coef()

In [ ]:
# Test 11: shuffle + continuous x + gf_coef
# Line and rise-over-run must both reflect the same shuffled data
set.seed(7)
gf_jitter(shuffle(Thumb) ~ Height, data = Fingers, width = 0.5, alpha = 0.4) %>%
  gf_lm() %>%
  gf_coef()

## Used inside gf_shuffle_grid()

In [ ]:
# Test 12: gf_shuffle_grid with categorical x (uses gf_lm internally)
options(repr.plot.width = 8, repr.plot.height = 6)
#gf_shuffle_grid(later_anxiety ~ condition, data = er, seed = 7)
gf_shuffle_grid(later_anxiety ~ condition, data = sample(er,12))

In [ ]:
# Test 13: gf_shuffle_grid with continuous x
gf_shuffle_grid(Thumb ~ Height, data = sample(Fingers,40), plot = "point", reveal = TRUE)

In [ ]:
# Test 14: gf_shuffle_grid with show_coef = TRUE
gf_shuffle_grid(later_anxiety ~ condition, data = sample(er, 20),
                show_model = TRUE, show_coef = TRUE, seed = 7)

## Standalone passthrough (should behave identically to ggformula::gf_lm)

In [ ]:
# Test 15: standalone — object is a formula, not a plot
# Should produce a plot with a regression line just like the original gf_lm()
gf_lm(Thumb ~ Height, data = Fingers)


In [ ]:
# Test 16: standalone with confidence interval — passthrough arg to ggformula::gf_lm
gf_point(Thumb ~ Height, data = Fingers, alpha = 0.4) %>%
  ggformula::gf_lm(interval = "confidence")   # explicit namespace to confirm original still works

## Edge cases

In [ ]:
# Test 17: x is an explicit factor() call in the formula
gf_jitter(Thumb ~ factor(Year), data = Fingers, width = 0.1, alpha = 0.4) %>%
  gf_lm()

In [ ]:
# Test 18: custom color flows through to segments (categorical)
gf_jitter(later_anxiety ~ condition, data = er, width = 0.1, alpha = 0.4) %>%
  gf_lm(color = "steelblue", linewidth = 1.5)

In [ ]:
# Test 19: calling gf_lm() then gf_lm_cat() — second freeze should be a no-op
# Both should show the same segments
set.seed(7)
p <- gf_jitter(shuffle(later_anxiety) ~ condition, data = er, width = 0.1, alpha = 0.4) %>%
  gf_lm()

# gf_lm_cat on an already-frozen plot — segments should be identical
p %>% gf_lm_cat(color = "tomato")

In [ ]:
# Test 20: reproducibility — same seed gives same model, even if jitter positions differ
# The jitter dots will look different between p1 and p2 (jitter is re-randomized
# each time gf_jitter() is called). What should be identical is the group-mean
# SEGMENTS — because set.seed() fixes the shuffle, so both models are fit on the
# same permuted y values.
set.seed(42)
p1 <- gf_jitter(shuffle(later_anxiety) ~ condition, data = er, width = 0.1, alpha = 0.4) %>%
  gf_lm()

set.seed(42)
p2 <- gf_jitter(shuffle(later_anxiety) ~ condition, data = er, width = 0.1, alpha = 0.4) %>%
  gf_lm()

# Jitter positions differ; segment heights should be identical
p1
p2

In [ ]:
# Test 21: passthrough of lm formula args to StatLm (continuous x only)
# Both syntaxes ggformula supports should still work when piped

# Polynomial fit via formula param
gf_point(Thumb ~ Height, data = Fingers, alpha = 0.4) %>%
  gf_lm(formula = y ~ poly(x, 2))

# Same via lm.args
gf_point(Thumb ~ Height, data = Fingers, alpha = 0.4) %>%
  gf_lm(lm.args = list(formula = y ~ poly(x, 2)))

In [ ]:
# Test 22: a polynomial data pattern

x <- rnorm(n=100, mean = 0, sd = 1)
y <- x^2 + rnorm(n=100, mean = 0, sd = .25)
dat <- data.frame(x, y)
gf_point(y ~ x, data = dat) %>%
  gf_lm(formula = y ~ poly(x, 2))

x <- rnorm(n=100, mean = 0, sd = 1)
y <- x^3 + rnorm(n=100, mean = 0, sd = .5)
dat <- data.frame(x, y)

gf_point(y ~ x, data = dat) %>%
  gf_lm(formula = y ~ poly(x, 2))

gf_point(y ~ x, data = dat) %>%
  gf_lm(formula = y ~ poly(x, 3))

gf_point(y ~ x, data = dat) %>%
  gf_lm(formula = y ~ poly(x, 6))